# 🌾 Egypt Farming Decision Support Agent
A LangGraph agentic workflow that combines:
- **RAG** — searches local agricultural documents
- **Weather API** — real-time conditions via wttr.in
- **Web Search** — pest alerts and farming tips via Tavily
- **SQLite Memory** — preserves crop/location context across conversations
- **Routing Logic** — conditional edges decide which tools to run

## 1. Install Dependencies

In [86]:
# Run once to install all required packages
#! pip install langchain langchain-community langchain-google-genai
#! pip install langgraph langgraph-checkpoint-sqlite
#! pip install chromadb tavily-python requests python-dotenv
#! pip install -q -U langchain-google-genai
#! pip install -q google-generativeai
#! pip install langchain-ollama
#! pip install pinecone langchain-pinecone

# ollama pull qwen3:1.7b

## 2. Imports & Environment Setup

In [87]:
import os
import operator
import requests
from typing import Literal
from typing_extensions import TypedDict, Annotated

from dotenv import load_dotenv
load_dotenv()

# LangChain core
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_ollama import ChatOllama

# LangChain community
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langgraph.checkpoint.memory import MemorySaver

# LangGraph
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver

from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

# LLM — using Google Gemini (free tier)
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

print("✅ All imports successful")

✅ All imports successful


## 3. LLM & Embeddings Initialization

In [88]:
# LLM used in all nodes via LCEL chains
llm = ChatOllama(model="qwen3:1.7b", temperature=0)

# Embeddings for RAG
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ LLM and embeddings ready")

# import google.generativeai as genai
# import os

# genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
# for m in genai.list_models():
#     if "embedContent" in m.supported_generation_methods:
#         print(m.name)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ LLM and embeddings ready


## 4. Agent State Definition

This is the **shared memory** passed between all nodes in the graph.
- `messages` uses `add_messages` reducer so conversation history accumulates (never overwritten)
- `crop` and `location` are plain strings that persist via SQLite checkpointer across sessions
- Tool results are plain strings written by each tool node

In [89]:
class AgentState(TypedDict):
    # Conversation history — accumulates with add_messages reducer
    messages: Annotated[list[BaseMessage], add_messages]

    # Extracted context — persisted across turns via SQLite checkpointer
    crop: str
    location: str

    # Tool outputs — written by each tool node
    rag_results: str
    weather_results: str
    web_results: str

    # Final synthesized response
    final_answer: str

    # Routing flags set by the router node
    needs_rag: bool
    needs_weather: bool
    needs_web: bool

    # Quality check flag for the self-correction loop
    answer_is_sufficient: bool

print("✅ AgentState defined")

✅ AgentState defined


## 5. RAG Setup — Build the Vector Store

Run this cell **once** to load your `diseases.txt` document, chunk it, and store embeddings in ChromaDB.
After that, the vector store is loaded from disk.

In [90]:
# Init Pinecone
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

INDEX_NAME = "crop-index"

# Create index once if it doesn't exist
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,          # all-MiniLM-L6-v2 output size
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"✅ Pinecone index '{INDEX_NAME}' created")

def build_vectorstore():
    loader = PyPDFDirectoryLoader("pdfs/")
    docs = loader.load()
    print(f"📄 Loaded {len(docs)} pages from PDFs")

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(docs)
    print(f"✂️ Created {len(chunks)} chunks")

    vectordb = PineconeVectorStore.from_documents(
        chunks,
        embeddings,
        index_name=INDEX_NAME
    )
    print(f"✅ Vector store built with {len(chunks)} chunks")
    return vectordb

def load_vectorstore():
    return PineconeVectorStore.from_existing_index(
        index_name=INDEX_NAME,
        embedding=embeddings
    )

# Check if index already has vectors
index = pc.Index(INDEX_NAME)
if index.describe_index_stats()["total_vector_count"] == 0:
    vectordb = build_vectorstore()
else:
    vectordb = load_vectorstore()
    print("✅ Vector store loaded from Pinecone")

retriever = vectordb.as_retriever(search_kwargs={"k": 3})
print("✅ Retriever ready")

✅ Vector store loaded from Pinecone
✅ Retriever ready


## 6. Graph Nodes

Each node is a pure function `(state) -> dict` that returns **only the keys it updates**.
Nodes use **LCEL chains** (`prompt | llm | parser`) as required.

### Node 1 — Extract Info
Reads the latest user message and extracts `crop` and `location` using LCEL.

In [91]:
def extract_info(state: AgentState) -> dict:
    """
    Node 1: Extract crop and location from the latest user message.
    Uses LCEL: prompt | llm | parser
    If already known from a previous turn (via SQLite memory), keeps them.
    """
    latest_message = state["messages"][-1].content

    # Build context string if we already have crop/location from memory
    memory_context = ""
    if state.get("crop") or state.get("location"):
        memory_context = (
            f"Previously known context — Crop: {state.get('crop', 'unknown')}, "
            f"Location: {state.get('location', 'unknown')}. "
            "Use these if the user doesn't mention them again."
        )

    # LCEL chain for extraction
    extract_prompt = ChatPromptTemplate.from_messages([
        SystemMessage(content=(
            "You are an agricultural assistant. Extract the crop name and Egyptian location "
            "from the user's message. " + memory_context + "\n"
            "Reply in this exact format:\n"
            "CROP: <crop name or 'unknown'>\n"
            "LOCATION: <location or 'unknown'>"
        )),
        ("human", "{message}")
    ])

    extract_chain = extract_prompt | llm | StrOutputParser()
    result = extract_chain.invoke({"message": latest_message})

    # Parse the structured output
    crop = state.get("crop", "unknown")
    location = state.get("location", "unknown")

    for line in result.strip().split("\n"):
        if line.startswith("CROP:"):
            extracted = line.replace("CROP:", "").strip()
            if extracted and extracted.lower() != "unknown":
                crop = extracted
        elif line.startswith("LOCATION:"):
            extracted = line.replace("LOCATION:", "").strip()
            if extracted and extracted.lower() != "unknown":
                location = extracted

    print(f"  [extract_info] crop={crop}, location={location}")
    return {"crop": crop, "location": location}

print("✅ Node 1 (extract_info) defined")

✅ Node 1 (extract_info) defined


### Node 2 — Router
This is the **routing logic** node. It decides which tools to run based on the query,
by setting boolean flags in the state. This makes the routing fully transparent.

In [92]:
def router_node(state: AgentState) -> dict:
    """
    Node 2: Decide which tools are needed for this query.
    Uses LCEL to classify the query, then sets routing flags.
    This is the explicit routing logic — NOT a black box.
    """
    latest_message = state["messages"][-1].content.lower()

    # LCEL routing chain
    router_prompt = ChatPromptTemplate.from_messages([
        SystemMessage(content=(
            "You are a routing assistant for an agriculture support system. "
            "Decide which tools are needed to answer the user's message. "
            "Reply with ONLY a comma-separated list of needed tools from: rag, weather, web.\n"
            "Rules:\n"
            "- rag: if the question is about crop diseases, pests, treatment, or agricultural knowledge\n"
            "- weather: if the question mentions weather, temperature, humidity, or farming conditions\n"
            "- web: if the question is about current pest alerts, recent news, or market prices\n"
            "Example: rag,weather  or  rag  or  rag,weather,web"
        )),
        ("human", "{message}")
    ])

    routing_chain = router_prompt | llm | StrOutputParser()
    decision = routing_chain.invoke({"message": latest_message})

    tools_needed = [t.strip() for t in decision.lower().split(",")]

    needs_rag = "rag" in tools_needed
    needs_weather = "weather" in tools_needed
    needs_web = "web" in tools_needed

    # If the crop is unknown, always use RAG to try to find info
    if state.get("crop", "unknown") == "unknown":
        needs_rag = True

    print(f"  [router] needs_rag={needs_rag}, needs_weather={needs_weather}, needs_web={needs_web}")
    return {
        "needs_rag": needs_rag,
        "needs_weather": needs_weather,
        "needs_web": needs_web,
        # Reset tool outputs for this new turn
        "rag_results": "",
        "weather_results": "",
        "web_results": "",
        "answer_is_sufficient": False
    }

print("✅ Node 2 (router_node) defined")

✅ Node 2 (router_node) defined


### Node 3 — RAG Tool
Searches agricultural documents using LCEL RAG chain.

In [93]:
def rag_node(state: AgentState) -> dict:
    """
    Node 3: Retrieval-Augmented Generation from local agricultural documents.
    Uses LCEL: retriever | format_docs → prompt | llm | parser
    Only runs if needs_rag=True (set by router).
    """
    if not state.get("needs_rag", False):
        print("  [rag_node] skipped (not needed)")
        return {"rag_results": "Not needed for this query."}

    query = f"{state.get('crop', '')} {state['messages'][-1].content}"

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # LCEL RAG chain
    rag_prompt = ChatPromptTemplate.from_template(
        """You are an agricultural expert. Answer ONLY using the provided context. If the context doesn't contain enough information, say 'I don't have enough data on this' — do NOT use outside knowledge.
Context from agricultural documents:
{context}

Question: {question}

Provide relevant disease/pest information, symptoms, and treatments:"""
    )

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | rag_prompt
        | llm
        | StrOutputParser()
    )

    result = rag_chain.invoke(query)
    print(f"  [rag_node] retrieved info ({len(result)} chars)")
    return {"rag_results": result}

print("✅ Node 3 (rag_node) defined")

✅ Node 3 (rag_node) defined


### Node 4 — Weather Tool
Fetches real-time weather from wttr.in API.

In [94]:
def weather_node(state: AgentState) -> dict:
    """
    Node 4: Fetch real-time weather from wttr.in for the user's location.
    Only runs if needs_weather=True (set by router).
    """
    if not state.get("needs_weather", False):
        print("  [weather_node] skipped (not needed)")
        return {"weather_results": "Not needed for this query."}

    location = state.get("location", "Cairo").replace(" ", "+")

    try:
        url = f"https://wttr.in/{location}?format=j1"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()

        current = data["current_condition"][0]
        weather_summary = (
            f"Location: {location}\n"
            f"Temperature: {current['temp_C']}°C\n"
            f"Feels like: {current['FeelsLikeC']}°C\n"
            f"Humidity: {current['humidity']}%\n"
            f"Description: {current['weatherDesc'][0]['value']}\n"
            f"Wind speed: {current['windspeedKmph']} km/h"
        )
        print(f"  [weather_node] fetched weather for {location}")
        return {"weather_results": weather_summary}

    except requests.exceptions.ConnectionError:
        return {"weather_results": "Weather unavailable: No internet connection"}
    except requests.exceptions.Timeout:
        return {"weather_results": "Weather unavailable: Request timed out"}
    except Exception as e:
        return {"weather_results": f"Weather unavailable: {e}"}

print("✅ Node 4 (weather_node) defined")

✅ Node 4 (weather_node) defined


### Node 5 — Web Search Tool
Searches for current pest alerts and farming tips using Tavily.

In [95]:
search_tool = TavilySearchResults(max_results=3)

def web_search_node(state: AgentState) -> dict:
    """
    Node 5: Tavily web search for current pest alerts and farming tips.
    Only runs if needs_web=True (set by router).
    """
    if not state.get("needs_web", False):
        print("  [web_search_node] skipped (not needed)")
        return {"web_results": "Not needed for this query."}

    crop = state.get("crop", "crop")
    location = state.get("location", "Egypt")
    query = f"{crop} pest alert disease treatment Egypt {location} agriculture 2024"

    try:
        results = search_tool.invoke(query)
        formatted = "\n".join(
            f"- {r.get('title', 'No title')}: {r.get('content', '')[:200]}"
            for r in results
        )
        print(f"  [web_search_node] found {len(results)} results")
        return {"web_results": formatted}
    except Exception as e:
        return {"web_results": f"Web search unavailable: {e}"}

print("✅ Node 5 (web_search_node) defined")

✅ Node 5 (web_search_node) defined


### Node 6 — Diagnosis & Synthesis
Combines all tool results into a final context-aware answer using LCEL.

In [96]:
def diagnosis_node(state: AgentState) -> dict:
    """
    Node 6: Synthesize all gathered information into a diagnosis and recommendations.
    Uses LCEL: prompt | llm | parser
    """
    # LCEL synthesis chain
    synthesis_prompt = ChatPromptTemplate.from_messages([
        SystemMessage(content=(
            "You are an expert Egyptian agricultural advisor. "
            "Based on the information provided, give a clear diagnosis and actionable recommendations. "
            "Be specific, practical, and consider Egyptian farming conditions. "
            "Respond in the same language the user used (Arabic or English)."
        )),
        ("human", (
            "Farmer's question: {user_question}\n\n"
            "Crop: {crop}\n"
            "Location: {location}\n\n"
            "--- Agricultural Knowledge (from documents) ---\n{rag_results}\n\n"
            "--- Current Weather Conditions ---\n{weather_results}\n\n"
            "--- Current Pest Alerts & Tips (from web) ---\n{web_results}\n\n"
            "Provide a diagnosis and step-by-step recommendations:"
        ))
    ])

    synthesis_chain = synthesis_prompt | llm | StrOutputParser()

    answer = synthesis_chain.invoke({
        "user_question": state["messages"][-1].content,
        "crop": state.get("crop", "unknown"),
        "location": state.get("location", "unknown"),
        "rag_results": state.get("rag_results", "No data."),
        "weather_results": state.get("weather_results", "No data."),
        "web_results": state.get("web_results", "No data."),
    })

    print(f"  [diagnosis_node] answer generated ({len(answer)} chars)")
    return {
        "final_answer": answer,
        "messages": [AIMessage(content=answer)]
    }

print("✅ Node 6 (diagnosis_node) defined")

✅ Node 6 (diagnosis_node) defined


### Node 7 — Quality Check (Self-Correction Loop)
Evaluates if the answer is sufficient. If not, it triggers a web search retry.

In [97]:
def quality_check_node(state: AgentState) -> dict:
    """
    Node 7: Check if the diagnosis is sufficient.
    If the answer is vague or says 'no information found', mark for retry.
    Uses LCEL: prompt | llm | parser
    """
    check_prompt = ChatPromptTemplate.from_messages([
        SystemMessage(content=(
            "You are a quality reviewer. Judge if this agricultural advice is specific and actionable. "
            "Reply with ONLY: SUFFICIENT or INSUFFICIENT"
        )),
        ("human", "Answer to review:\n{answer}")
    ])

    quality_chain = check_prompt | llm | StrOutputParser()
    verdict = quality_chain.invoke({"answer": state.get("final_answer", "")})

    is_sufficient = "SUFFICIENT" in verdict.upper()
    print(f"  [quality_check] verdict={verdict.strip()}, is_sufficient={is_sufficient}")

    # If not sufficient, force web search in the retry
    if not is_sufficient:
        return {
            "answer_is_sufficient": False,
            "needs_web": True  # force web search on retry
        }

    return {"answer_is_sufficient": True}

print("✅ Node 7 (quality_check_node) defined")

✅ Node 7 (quality_check_node) defined


## 7. Routing Functions (Conditional Edges)

These are pure routing functions — they read state flags and return the next node name.
**No LLM calls here** — routing is deterministic and transparent.

In [98]:
def route_after_extract(state: AgentState) -> Literal["router", "ask_clarification"]:
    """
    After extraction: if we have at least a crop OR location, proceed.
    Otherwise ask the user for clarification.
    """
    has_crop = state.get("crop", "unknown") != "unknown"
    has_location = state.get("location", "unknown") != "unknown"

    if has_crop or has_location:
        return "router"
    else:
        return "ask_clarification"


def route_after_rag(state: AgentState) -> Literal["weather", "web", "diagnosis"]:
    """
    After RAG: check if weather or web search is also needed.
    """
    if state.get("needs_weather", False):
        return "weather"
    elif state.get("needs_web", False):
        return "web"
    else:
        return "diagnosis"


def route_after_weather(state: AgentState) -> Literal["web", "diagnosis"]:
    """
    After weather: check if web search is also needed.
    """
    if state.get("needs_web", False):
        return "web"
    else:
        return "diagnosis"


def route_after_quality(state: AgentState) -> Literal["diagnosis", END]:
    """
    After quality check:
    - If sufficient → END
    - If not sufficient → loop back to diagnosis (with web search forced ON)
    This is the self-correction loop.
    """
    if state.get("answer_is_sufficient", True):
        return END
    else:
        return "web"  # retry with forced web search


print("✅ All routing functions defined")

✅ All routing functions defined


### Clarification Node
If the user didn't provide crop or location, ask them.

In [99]:
def ask_clarification(state: AgentState) -> dict:
    """
    Fallback node: Ask the user to provide crop and location.
    """
    message = (
        "I need a bit more information to help you. "
        "Could you please tell me:\n"
        "1. What crop are you growing?\n"
        "2. What is your location in Egypt?"
    )
    print("  [ask_clarification] requesting more info from user")
    return {
        "final_answer": message,
        "messages": [AIMessage(content=message)]
    }

print("✅ Clarification node defined")

✅ Clarification node defined


## 8. Build the LangGraph

Wire all nodes together with edges and conditional edges.

In [100]:
# Build the state graph
graph = StateGraph(AgentState)

# ── Add all nodes ──────────────────────────────────────────
graph.add_node("extract", extract_info)
graph.add_node("router", router_node)
graph.add_node("rag", rag_node)
graph.add_node("weather", weather_node)
graph.add_node("web", web_search_node)
graph.add_node("diagnosis", diagnosis_node)
graph.add_node("quality_check", quality_check_node)
graph.add_node("ask_clarification", ask_clarification)

# ── Add edges ──────────────────────────────────────────────

# Entry point
graph.add_edge(START, "extract")

# After extraction: route based on whether we have crop/location
graph.add_conditional_edges(
    "extract",
    route_after_extract,
    {"router": "router", "ask_clarification": "ask_clarification"}
)

# Router always goes to RAG first (RAG node skips itself if not needed)
graph.add_edge("router", "rag")

# After RAG: route to weather, web, or directly to diagnosis
graph.add_conditional_edges(
    "rag",
    route_after_rag,
    {"weather": "weather", "web": "web", "diagnosis": "diagnosis"}
)

# After weather: route to web or diagnosis
graph.add_conditional_edges(
    "weather",
    route_after_weather,
    {"web": "web", "diagnosis": "diagnosis"}
)

# Web always goes to diagnosis
graph.add_edge("web", "diagnosis")

# Diagnosis goes to quality check
graph.add_edge("diagnosis", "quality_check")

# Quality check: either END (sufficient) or retry via web → diagnosis (self-correction loop)
graph.add_conditional_edges(
    "quality_check",
    route_after_quality,
    {END: END, "web": "web"}
)

# Clarification ends the turn
graph.add_edge("ask_clarification", END)

print("✅ Graph structure built")
print("""
Graph flow:
START → extract
         ↓ (has info?)      ↓ (no info)
       router          ask_clarification → END
         ↓
        rag
         ↓ (needs weather?) ↓ (needs web?) ↓ (direct)
       weather             web           diagnosis
         ↓ (needs web?) ↓ (direct)        ↑
        web →──────────────────────────────┘
         ↓
      diagnosis → quality_check
                     ↓ sufficient      ↓ not sufficient (loop)
                    END              web → diagnosis
""")

✅ Graph structure built

Graph flow:
START → extract
         ↓ (has info?)      ↓ (no info)
       router          ask_clarification → END
         ↓
        rag
         ↓ (needs weather?) ↓ (needs web?) ↓ (direct)
       weather             web           diagnosis
         ↓ (needs web?) ↓ (direct)        ↑
        web →──────────────────────────────┘
         ↓
      diagnosis → quality_check
                     ↓ sufficient      ↓ not sufficient (loop)
                    END              web → diagnosis



## 9. Compile with SQLite Memory

The `SqliteSaver` checkpointer persists the full state (including `crop` and `location`)
to a SQLite database. Every conversation turn is saved under a `thread_id`,
so context is preserved across sessions.

In [101]:
# DB_PATH = "farming_agent_memory.sqlite3"

# # SqliteSaver is used as a context manager
# # We keep it open for the entire session
# memory = MemorySaver()

# app = graph.compile(checkpointer=memory)

# print(f"✅ Agent compiled with SQLite memory at: {DB_PATH}")
# print("   Crop/location context will be preserved across conversations.")

DB_PATH = "farming_agent_memory.sqlite3"
conn = sqlite3.connect(DB_PATH, check_same_thread=False)
memory = SqliteSaver(conn)

app = graph.compile(checkpointer=memory)
print(f"✅ Agent compiled with SQLite memory at: {DB_PATH}")

✅ Agent compiled with SQLite memory at: farming_agent_memory.sqlite3


## 10. Run the Agent

Each conversation uses a `thread_id` — same thread = shared memory.
Different threads = separate users/sessions.

In [102]:
def chat(user_message: str, thread_id: str = "farmer-1") -> str:
    """
    Send a message to the farming agent and get a response.
    The thread_id preserves crop/location memory across calls.
    """
    config = {"configurable": {"thread_id": thread_id}}

    # Get current state to carry forward existing crop/location
    current_state = app.get_state(config)
    existing_crop = current_state.values.get("crop", "unknown") if current_state.values else "unknown"
    existing_location = current_state.values.get("location", "unknown") if current_state.values else "unknown"

    result = app.invoke(
        {
            "messages": [HumanMessage(content=user_message)],
            "crop": existing_crop,
            "location": existing_location,
            "rag_results": "",
            "weather_results": "",
            "web_results": "",
            "final_answer": "",
            "needs_rag": False,
            "needs_weather": False,
            "needs_web": False,
            "answer_is_sufficient": False
        },
        config
    )

    return result["final_answer"]

print("✅ chat() helper ready")

✅ chat() helper ready


### Test 1 — First message with full context

In [103]:
print("=" * 60)
print("TEST 1: Full context in one message")
print("=" * 60)

response = chat(
    "My tomatoes in Assiut have yellow leaves and white spots. What disease is this?",
    thread_id="farmer-1"
)

print("\n🌾 AGENT RESPONSE:")
print(response)

TEST 1: Full context in one message
  [extract_info] crop=tomatoes, location=Assiut
  [router] needs_rag=True, needs_weather=False, needs_web=False
  [rag_node] retrieved info (958 chars)
  [diagnosis_node] answer generated (2407 chars)
  [quality_check] verdict=SUFFICIENT, is_sufficient=True

🌾 AGENT RESPONSE:
**Diagnosis**: The symptoms (yellow leaves with white spots) are consistent with **early blight (Alternaria solani)**, a fungal disease common in humid or moist environments. The white spots may result from fungal spores or secondary infections, while the yellowing is a hallmark of the disease.  

---

### **Actionable Recommendations**  
1. **Remove and destroy infected plant material**:  
   - Cut down infected leaves, stems, and fruits.  
   - Burn or dispose of infected parts to prevent spread.  

2. **Improve air circulation**:  
   - Stake plants vertically to allow airflow.  
   - Space plants appropriately (e.g., 60–80 cm apart) to reduce humidity buildup.  

3. **Water 

### Test 2 — Follow-up question (memory test)
This tests that crop and location are remembered from the previous turn.

In [104]:
print("=" * 60)
print("TEST 2: Follow-up (memory test — no crop/location mentioned)")
print("=" * 60)

response = chat(
    "What is the weather like and will it make the disease worse?",
    thread_id="farmer-1"  # Same thread = same memory
)

print("\n🌾 AGENT RESPONSE:")
print(response)

TEST 2: Follow-up (memory test — no crop/location mentioned)
  [extract_info] crop=tomatoes, location=Assiut
  [router] needs_rag=True, needs_weather=True, needs_web=False
  [rag_node] retrieved info (1108 chars)
  [weather_node] fetched weather for Assiut
  [diagnosis_node] answer generated (1993 chars)
  [quality_check] verdict=SUFFICIENT, is_sufficient=True

🌾 AGENT RESPONSE:
**Diagnosis**:  
The current weather in Assiut (40°C, 9% humidity, sunny, 14 km/h wind) reflects extreme heat and low humidity, which can stress tomato plants. While the weather is dry, the high temperature (40°C) and low humidity (9%) may exacerbate disease risk, particularly for **tomato late blight** (a fungal disease thrives in warm, humid conditions). The combination of heat stress and dry air can weaken plant immunity, making them more susceptible to pathogens.  

---

**Actionable Recommendations**:  
1. **Fungicide Application**:  
   - Apply **mancozeb** or **benomyl** fungicides (or equivalent) to fol

### Test 3 — Missing info (triggers clarification node)

In [105]:
print("=" * 60)
print("TEST 3: Missing crop and location → clarification node")
print("=" * 60)

response = chat(
    "My plants are sick help me",
    thread_id="farmer-new"  # New thread = no memory
)

print("\n🌾 AGENT RESPONSE:")
print(response)

TEST 3: Missing crop and location → clarification node
  [extract_info] crop=unknown, location=unknown
  [ask_clarification] requesting more info from user

🌾 AGENT RESPONSE:
I need a bit more information to help you. Could you please tell me:
1. What crop are you growing?
2. What is your location in Egypt?


### Test 4 — Web search query

In [106]:
print("=" * 60)
print("TEST 4: Web search triggered")
print("=" * 60)

response = chat(
    "Are there any current pest alerts for wheat in Fayoum this season?",
    thread_id="farmer-2"
)

print("\n🌾 AGENT RESPONSE:")
print(response)

TEST 4: Web search triggered
  [extract_info] crop=wheat, location=Fayoum
  [router] needs_rag=False, needs_weather=False, needs_web=True
  [rag_node] skipped (not needed)
  [web_search_node] found 3 results
  [diagnosis_node] answer generated (1851 chars)
  [quality_check] verdict=SUFFICIENT, is_sufficient=True

🌾 AGENT RESPONSE:
**Diagnosis:**  
There are no specific pest alerts for wheat in Fayoum this season, but common pests such as aphids, wheat rust, and stem borers are likely present. The region's climate and agricultural practices may exacerbate these issues, particularly during the growing season.  

---

**Step-by-Step Recommendations:**  

1. **Monitor Pest Activity:**  
   - Regularly inspect wheat fields for signs of pests (e.g., aphids, rust spots, or borers).  
   - Use traps or visual inspections to track population levels.  

2. **Implement Biological Controls:**  
   - Introduce natural predators (e.g., ladybugs, parasitic wasps) or use neem oil (a natural pesticide)

## 11. Inspect Memory State (Optional Debug)

In [107]:
# See what is currently stored in memory for farmer-1
config = {"configurable": {"thread_id": "farmer-1"}}
state = app.get_state(config)

if state.values:
    print("📦 Memory for farmer-1:")
    print(f"  Crop: {state.values.get('crop', 'not set')}")
    print(f"  Location: {state.values.get('location', 'not set')}")
    print(f"  Messages in history: {len(state.values.get('messages', []))}")
    print(f"  Last answer preview: {state.values.get('final_answer', '')[:100]}...")
else:
    print("No state found for farmer-1")

📦 Memory for farmer-1:
  Crop: tomatoes
  Location: Assiut
  Messages in history: 4
  Last answer preview: **Diagnosis**:  
The current weather in Assiut (40°C, 9% humidity, sunny, 14 km/h wind) reflects ext...


In [108]:
index = pc.Index(INDEX_NAME)
stats = index.describe_index_stats()
print(stats)
print("Total vectors:", stats["total_vector_count"])

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 1325}},
 'total_vector_count': 1325,
 'vector_type': 'dense'}
Total vectors: 1325
